 ### Data loading and initial inspection

This analysis uses three files: `customers.csv`, `articles.csv`, and `transactions_train.csv`. Initially, I loaded five rows from each file to inspect their structure and confirm that they could be accessed correctly. I did not load the full transactions file because it contains more than 31 million rows and could exceed the available memory. Each row in `transactions_train.csv` represents one article purchased by one customer on a specific date through a sales channel; it does not necessarily represent a complete order. The `price` column contains normalized numerical values, but its currency and scaling method are not documented. Therefore, it will be used only for relative comparisons and will not be interpreted as revenue in euros.


In [1]:
import pandas as pd
import numpy as np
import pathlib as path

In [2]:
data_dir = path.Path.home() / "Downloads" / "hm_data"

In [3]:
data_dir.exists()

True

In [4]:
list(data_dir.iterdir())

[PosixPath('/Users/anastasiia/Downloads/hm_data/customers.csv'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/customers.csv.zip'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/.DS_Store'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/articles.csv'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/articles.csv.zip'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/transactions_train.csv'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/transactions_train.csv.zip')]

In [5]:
customers_sample = pd.read_csv(data_dir/"customers.csv", nrows=5)

In [6]:
customers_sample.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [7]:
articles_sample = pd.read_csv(data_dir/"articles.csv", nrows=5)

In [8]:
articles_sample.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [9]:
transactions_sample = pd.read_csv(data_dir/"transactions_train.csv",nrows=5)

In [10]:
transactions_sample.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [11]:
transactions_sample.nunique()

t_dat               1
customer_id         2
article_id          5
price               4
sales_channel_id    1
dtype: int64

In [12]:
purchase_ocassions_sample = transactions_sample[["t_dat","customer_id","sales_channel_id"]].drop_duplicates()

In [13]:
purchase_ocassions_sample.shape

(2, 3)

In [14]:
transactions_sample.dtypes

t_dat                   str
customer_id             str
article_id            int64
price               float64
sales_channel_id      int64
dtype: object

In [15]:
transactions_sample["t_dat"] = pd.to_datetime(transactions_sample["t_dat"])

In [16]:
transactions_sample.dtypes

t_dat               datetime64[us]
customer_id                    str
article_id                   int64
price                      float64
sales_channel_id             int64
dtype: object

In [17]:
transactions_path = data_dir / "transactions_train.csv"

In [18]:
transactions_path.exists()

True

In [19]:
transactions_path.stat().st_size

3488002253

# Convert the transactions file size from bytes to gigabytes

In [20]:
transactions_size_gb= transactions_path.stat().st_size/ 1024 ** 3

In [21]:
round(transactions_size_gb, 2)

3.25

In [22]:
import duckdb
duckdb.__version__

'1.5.5'

## DuckDB connection

DuckDB is used to query the complete transactions file efficiently without loading the entire CSV into pandas memory.

In [23]:
db_path = data_dir / "hm_retention.duckdb"

In [25]:
con = duckdb.connect(str(db_path))

In [26]:
con.sql("SELECT 42 AS connection_test").show()

┌─────────────────┐
│ connection_test │
│      int32      │
├─────────────────┤
│              42 │
└─────────────────┘



In [27]:
transactions_relation = con.read_csv(str(transactions_path))

In [28]:
transactions_relation.create_view("transactions", replace=True)

┌────────────┬──────────────────────────────────────────────────────────────────┬────────────┬───────────────────────┬──────────────────┐
│   t_dat    │                           customer_id                            │ article_id │         price         │ sales_channel_id │
│    date    │                             varchar                              │  varchar   │        double         │      int64       │
├────────────┼──────────────────────────────────────────────────────────────────┼────────────┼───────────────────────┼──────────────────┤
│ 2018-09-20 │ 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 │ 0663713001 │  0.050830508474576264 │                2 │
│ 2018-09-20 │ 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 │ 0541518023 │   0.03049152542372881 │                2 │
│ 2018-09-20 │ 00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2 │ 0505221004 │   0.01523728813559322 │                2 │
│ 2018-09-20 │ 00007d2de826758b65a

## Transaction sample audit

1. Each row represents one article purchased by a customer, not a complete order.
2. Five rows represent five articles, but they do not necessarily represent five orders. The sample contains two approximate purchase occasions, although the exact number of orders is unknown because there is no order ID.
3. The `price` column has a numeric `float64` data type. However, its currency and scale are not documented, so it cannot be interpreted as a price in euros.

## Full transactions audit with SQL

DuckDB is used to audit the complete transactions dataset without loading all rows into pandas memory.

In [29]:
con.sql("""
    SELECT *
    FROM transactions
    LIMIT 5
""").show()

┌────────────┬──────────────────────────────────────────────────────────────────┬────────────┬──────────────────────┬──────────────────┐
│   t_dat    │                           customer_id                            │ article_id │        price         │ sales_channel_id │
│    date    │                             varchar                              │  varchar   │        double        │      int64       │
├────────────┼──────────────────────────────────────────────────────────────────┼────────────┼──────────────────────┼──────────────────┤
│ 2018-09-20 │ 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 │ 0663713001 │ 0.050830508474576264 │                2 │
│ 2018-09-20 │ 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 │ 0541518023 │  0.03049152542372881 │                2 │
│ 2018-09-20 │ 00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2 │ 0505221004 │  0.01523728813559322 │                2 │
│ 2018-09-20 │ 00007d2de826758b65a93dd24c

In [30]:
con.sql("""
    DESCRIBE transactions
""").show()

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ t_dat            │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ article_id       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ sales_channel_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [31]:
con.sql("""
    SELECT
        COUNT(*) AS transactions_rows,
        COUNT(DISTINCT customer_id) AS unique_customer,
        COUNT(DISTINCT article_id) AS unique_article
    FROM transactions
""").show()

┌───────────────────┬─────────────────┬────────────────┐
│ transactions_rows │ unique_customer │ unique_article │
│       int64       │      int64      │     int64      │
├───────────────────┼─────────────────┼────────────────┤
│          31788324 │         1362281 │         104547 │
└───────────────────┴─────────────────┴────────────────┘



In [33]:
con.sql("""
    SELECT
        MIN(t_dat) AS start_date,
        MAX(t_dat) AS end_date,
        COUNT(DISTINCT t_dat) AS active_days
    FROM transactions
""").show()

┌────────────┬────────────┬─────────────┐
│ start_date │  end_date  │ active_days │
│    date    │    date    │    int64    │
├────────────┼────────────┼─────────────┤
│ 2018-09-20 │ 2020-09-22 │         734 │
└────────────┴────────────┴─────────────┘



In [36]:
con.sql("""
    SELECT
        COUNT(*) - COUNT(t_dat) AS missing_t_dat,
        COUNT(*) - COUNT(customer_id) AS missing_customer_id,
        COUNT(*) - COUNT(article_id) AS missing_article_id,
        COUNT(*) - COUNT(price) AS missing_price,
        COUNT(*) - COUNT(sales_channel_id) AS missing_sales_channel
    FROM transactions
""").show()

┌───────────────┬─────────────────────┬────────────────────┬───────────────┬───────────────────────┐
│ missing_t_dat │ missing_customer_id │ missing_article_id │ missing_price │ missing_sales_channel │
│     int64     │        int64        │       int64        │     int64     │         int64         │
├───────────────┼─────────────────────┼────────────────────┼───────────────┼───────────────────────┤
│             0 │                   0 │                  0 │             0 │                     0 │
└───────────────┴─────────────────────┴────────────────────┴───────────────┴───────────────────────┘



### Missing values finding

No missing values were found in any column of the transactions dataset. Therefore, no missing-value treatment is required before the retention analysis.

In [38]:
con.sql("""
    SELECT
        sales_channel_id,
        COUNT(*) AS transactions_rows
    FROM transactions
    GROUP BY sales_channel_id
    ORDER BY sales_channel_id
""").show()
        

┌──────────────────┬───────────────────┐
│ sales_channel_id │ transactions_rows │
│      int64       │       int64       │
├──────────────────┼───────────────────┤
│                1 │           9408462 │
│                2 │          22379862 │
└──────────────────┴───────────────────┘



In [39]:
con.sql("""
    SELECT
        sales_channel_id,
        COUNT(*) AS transaction_rows,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS transaction_share_pct
    FROM transactions
    GROUP BY sales_channel_id
    ORDER BY sales_channel_id
""").show()

┌──────────────────┬──────────────────┬───────────────────────┐
│ sales_channel_id │ transaction_rows │ transaction_share_pct │
│      int64       │      int64       │        double         │
├──────────────────┼──────────────────┼───────────────────────┤
│                1 │          9408462 │                  29.6 │
│                2 │         22379862 │                  70.4 │
└──────────────────┴──────────────────┴───────────────────────┘



### Sales channel finding

Channel 2 accounts for approximately 70.4% of all transaction rows, while Channel 1 accounts for 29.6%. These percentages represent transaction lines rather than customers or complete orders. Because the channel labels are not documented, the analysis retains the original Channel 1 and Channel 2 names.

In [40]:
con.sql("""
    SELECT
        COUNT(*) AS approximate_purchase_occasions
    FROM (
        SELECT DISTINCT
            customer_id,
            t_dat,
            sales_channel_id
        FROM transactions
    ) AS purchase_occasions
""").show()

┌────────────────────────────────┐
│ approximate_purchase_occasions │
│             int64              │
├────────────────────────────────┤
│                        9174457 │
└────────────────────────────────┘



### Approximate purchase occasions

The 31,788,324 transaction rows were reduced to 9,174,457 approximate purchase occasions using each unique combination of customer, date, and sales channel. These occasions are not confirmed orders because the dataset does not contain an order ID.

In [52]:
con.sql("""
    SELECT
        customer_id,
        t_dat,
        sales_channel_id,
        COUNT(*) transaction_line
    FROM transactions
    GROUP BY
          customer_id,
          t_dat,
          sales_channel_id
    ORDER BY transaction_lines_per_occasion DESC
    LIMIT 10 
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬──────────────────┬────────────────────────────────┐
│                           customer_id                            │   t_dat    │ sales_channel_id │ transaction_lines_per_occasion │
│                             varchar                              │    date    │      int64       │             int64              │
├──────────────────────────────────────────────────────────────────┼────────────┼──────────────────┼────────────────────────────────┤
│ d00063b94dcb1342869d4994844a2742b5d62927f36843164fb3f818f630bca9 │ 2018-10-14 │                1 │                            570 │
│ c2f0cdda2dc3042ccd9fcd8253fd8e368769840581e40aab1d87a64ff39987e3 │ 2018-12-17 │                2 │                            336 │
│ 62fd7d41b587c72a95c31eca5046019ce4e802853397ffd00f354c17007ebd0b │ 2019-01-13 │                2 │                            221 │
│ 246734d8f4a4252fcd5c7aa525055a2804b9a6fb3d4210e771a33ed98f2b

In [53]:
con.sql("""
    SELECT
        article_id,
        price,
        COUNT(*) AS total_rows,
        COUNT(*) AS repeated_rows,
        COUNT(DISTINCT article_id) AS unique_articles
    FROM transactions
    WHERE customer_id == 'd00063b94dcb1342869d4994844a2742b5d62927f36843164fb3f818f630bca9'
        AND t_dat = '2018-10-14'
        AND sales_channel_id = 1
    GROUP BY article_id,
             price
    ORDER BY total_rows
""").show()

┌────────────┬──────────────────────┬────────────┬───────────────┬─────────────────┐
│ article_id │        price         │ total_rows │ repeated_rows │ unique_articles │
│  varchar   │        double        │   int64    │     int64     │      int64      │
├────────────┼──────────────────────┼────────────┼───────────────┼─────────────────┤
│ 0678342001 │ 0.008694915254237287 │          1 │             1 │               1 │
│ 0678342001 │  0.00676271186440678 │        569 │           569 │               1 │
└────────────┴──────────────────────┴────────────┴───────────────┴─────────────────┘



### Repeated transaction lines

An extreme customer-date-channel combination contained 570 transaction rows but only one unique article. Of these rows, 569 had the same article and price. The dataset does not provide an order ID or quantity field, so these rows cannot be reliably classified as duplicated records or multiple units. Therefore, repeated rows will not be removed automatically. The main retention metric remains based on unique customer-date-channel occasions, while transaction-line counts and normalized value metrics will be interpreted cautiously.

In [56]:
con.sql("""
    SELECT
        customer_id,
        t_dat,
        sales_channel_id,
        COUNT(*) transaction_lines,
        COUNT(DISTINCT article_id) AS unique_articles
    FROM transactions
    GROUP BY
          customer_id,
          t_dat,
          sales_channel_id
    ORDER BY transaction_lines DESC
    LIMIT 10 
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬──────────────────┬───────────────────┬─────────────────┐
│                           customer_id                            │   t_dat    │ sales_channel_id │ transaction_lines │ unique_articles │
│                             varchar                              │    date    │      int64       │       int64       │      int64      │
├──────────────────────────────────────────────────────────────────┼────────────┼──────────────────┼───────────────────┼─────────────────┤
│ d00063b94dcb1342869d4994844a2742b5d62927f36843164fb3f818f630bca9 │ 2018-10-14 │                1 │               570 │               1 │
│ c2f0cdda2dc3042ccd9fcd8253fd8e368769840581e40aab1d87a64ff39987e3 │ 2018-12-17 │                2 │               336 │              45 │
│ 62fd7d41b587c72a95c31eca5046019ce4e802853397ffd00f354c17007ebd0b │ 2019-01-13 │                2 │               221 │              28 │
│ 246734d8f4a4252fcd5c7aa52

In [57]:
con.execute("""
    CREATE OR REPLACE TABLE purchase_occasions AS
    SELECT
        customer_id,
        t_dat,
        sales_channel_id,
        COUNT(*) transaction_lines,
        COUNT(DISTINCT article_id) AS unique_articles
    FROM transactions
    GROUP BY
          customer_id,
          t_dat,
          sales_channel_id
""")
    

In [60]:
con.sql("""
    SELECT
        COUNT(*) AS total_occasions
    FROM purchase_occasions
""").show()

┌─────────────────┐
│ total_occasions │
│      int64      │
├─────────────────┤
│         9174457 │
└─────────────────┘



In [63]:
con.sql("""
    SELECT
        ROUND(AVG(transaction_lines),2) AS avg_transactions_line,
        MEDIAN(transaction_lines) AS median_transactions_line,
        MAX(transaction_lines) AS max_transactions_line,
        ROUND(AVG(unique_articles),2) AS avg_unique_articles,
        MEDIAN(unique_articles) AS median_unique_articles,
        MAX(unique_articles) AS MAX_unique_articles
    FROM purchase_occasions
""").show()

┌───────────────────────┬──────────────────────────┬───────────────────────┬─────────────────────┬────────────────────────┬─────────────────────┐
│ avg_transactions_line │ median_transactions_line │ max_transactions_line │ avg_unique_articles │ median_unique_articles │ MAX_unique_articles │
│        double         │          double          │         int64         │       double        │         double         │        int64        │
├───────────────────────┼──────────────────────────┼───────────────────────┼─────────────────────┼────────────────────────┼─────────────────────┤
│                  3.46 │                      2.0 │                   570 │                3.12 │                    2.0 │                 159 │
└───────────────────────┴──────────────────────────┴───────────────────────┴─────────────────────┴────────────────────────┴─────────────────────┘



##  Purchase occasion size

A typical purchase occasion contains two unique articles. The average is higher than the median because a small number of extreme observations influence the mean. Therefore, the maximum values should not be used to describe typical customer behavior.


In [64]:
con.sql("""
    SELECT
        QUANTILE_CONT(transaction_lines, 0.95),
        QUANTILE_CONT(transaction_lines, 0.99),
        QUANTILE_CONT(unique_articles, 0.95),
        QUANTILE_CONT(unique_articles, 0.99)
    FROM purchase_occasions
""").show()

┌────────────────────────────────────────┬────────────────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┐
│ quantile_cont(transaction_lines, 0.95) │ quantile_cont(transaction_lines, 0.99) │ quantile_cont(unique_articles, 0.95) │ quantile_cont(unique_articles, 0.99) │
│                 double                 │                 double                 │                double                │                double                │
├────────────────────────────────────────┼────────────────────────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┤
│                                   10.0 │                                   18.0 │                                  9.0 │                                 15.0 │
└────────────────────────────────────────┴────────────────────────────────────────┴──────────────────────────────────────┴──────────────────────────────────────┘



In [67]:
con.sql("""
    SELECT
        customer_id,
        MIN(t_dat) AS first_observed_purchase_date
    FROM purchase_occasions
    GROUP BY customer_id
    LIMIT 10
""").show()

┌──────────────────────────────────────────────────────────────────┬──────────────────────────────┐
│                           customer_id                            │ first_observed_purchase_date │
│                             varchar                              │             date             │
├──────────────────────────────────────────────────────────────────┼──────────────────────────────┤
│ f0ae3d118a6b35a44065d5bbd3e40d45d1702d143b84f643dde4823f84da78e0 │ 2018-09-20                   │
│ 13ab0fc66b661342788759c4ee21f58fdfb72cbfd642b4e689c52c539fe08e4d │ 2019-02-22                   │
│ 1cce4d52abde898c9eb036e26ea0b4546740775ca703914f5ef4070456cc56c6 │ 2018-10-03                   │
│ 023d1b8bc0c4ebbc424fe52144370ee11743981a11f1e3fe8578ba672af1525e │ 2018-09-30                   │
│ 7cb753afc3904c6cdea76a64c7807bc9d28bf85211659d5ba5ee0b7a298fd3e2 │ 2018-11-10                   │
│ a2d2e0ad263b200c59de5420d963290011833f813579c7f756b6e748a8f375b2 │ 2019-03-16                   │


In [71]:
con.execute("""
    CREATE OR REPLACE TABLE customer_purchase_dates AS 
    SELECT DISTINCT
        customer_id,
        t_dat
    FROM purchase_occasions
""")
        

In [75]:
con.sql("""
    SELECT
        COUNT(*) AS total_customer_dates,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(DISTINCT t_dat) AS calendar_dates
    FROM customer_purchase_dates
""").show()

┌──────────────────────┬──────────────────┬────────────────┐
│ total_customer_dates │ unique_customers │ calendar_dates │
│        int64         │      int64       │     int64      │
├──────────────────────┼──────────────────┼────────────────┤
│              9080179 │          1362281 │            734 │
└──────────────────────┴──────────────────┴────────────────┘



In [93]:
con.execute("""
CREATE OR REPLACE TABLE customer_lifecycle AS
WITH  sequenced_purchase AS (
    SELECT
        customer_id,
        t_dat,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY t_dat) AS purchase_number,
        LEAD(t_dat) OVER(PARTITION BY customer_id ORDER BY t_dat) AS next_purchase_date
    FROM customer_purchase_dates
   
)
SELECT
    customer_id,
    t_dat AS first_date,
    next_purchase_date AS second_date,
    DATE_DIFF('day', t_dat, next_purchase_date) AS days_to_second,
    CASE
      WHEN t_dat <= DATE '2020-06-24' THEN 1
    ELSE 0
    END AS eligible_90d,
    CASE
    WHEN t_dat > DATE '2020-06-24' THEN NULL
    WHEN next_purchase_date IS NOT NULL
         AND DATE_DIFF('day', t_dat, next_purchase_date) <= 90 THEN 1
    ELSE 0
    END AS ret_90d
FROM sequenced_purchase
WHERE purchase_number = 1
""")

In [95]:
con.sql("""
    SELECT
        COUNT(*) AS total_customers,
        COUNT(DISTINCT customer_id) AS unique_customers
    FROM customer_lifecycle
""").show()

┌─────────────────┬──────────────────┐
│ total_customers │ unique_customers │
│      int64      │      int64       │
├─────────────────┼──────────────────┤
│         1362281 │          1362281 │
└─────────────────┴──────────────────┘



In [96]:
con.sql("""
DESCRIBE customer_lifecycle
""").show()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ first_date     │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ second_date    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ days_to_second │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ eligible_90d   │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ ret_90d        │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [118]:
con.sql("""
    SELECT
        SUM(
        CASE
          WHEN eligible_90d = 1 THEN 1
          ELSE 0
          END ) AS eligible_customers,
        SUM(
        CASE 
          WHEN eligible_90d= 1 AND ret_90d = 1 THEN  1
          ELSE 0
          END) AS retained_customers,
         ROUND(SUM(
        CASE 
          WHEN eligible_90d= 1 AND ret_90d = 1 THEN  1
          ELSE 0
          END ) /SUM(
        CASE
          WHEN eligible_90d = 1 THEN 1
          ELSE 0
          END )  * 100.0 ,2) AS retention_90d
    FROM customer_lifecycle

""").show()
    
          

┌────────────────────┬────────────────────┬───────────────┐
│ eligible_customers │ retained_customers │ retention_90d │
│       int128       │       int128       │    double     │
├────────────────────┼────────────────────┼───────────────┤
│            1291147 │             607798 │         47.07 │
└────────────────────┴────────────────────┴───────────────┘



In [121]:
con.sql("""
    SELECT
        strftime(first_date, '%Y-%m') AS first_month,
        COUNT(eligible_90d) AS eligible_customers,
        SUM(ret_90d) AS retained_customers,
        ROUND(100.0 * AVG(ret_90d),2) AS retention_90d
    FROM customer_lifecycle
    WHERE eligible_90d = 1
    GROUP BY first_month
    ORDER BY first_month
   
    
""").show()


┌─────────────┬────────────────────┬────────────────────┬───────────────┐
│ first_month │ eligible_customers │ retained_customers │ retention_90d │
│   varchar   │       int64        │       int128       │    double     │
├─────────────┼────────────────────┼────────────────────┼───────────────┤
│ 2018-09     │             140340 │              97553 │         69.51 │
│ 2018-10     │             212669 │             133579 │         62.81 │
│ 2018-11     │             138233 │              72018 │          52.1 │
│ 2018-12     │              89944 │              41237 │         45.85 │
│ 2019-01     │              68957 │              29581 │          42.9 │
│ 2019-02     │              65453 │              25782 │         39.39 │
│ 2019-03     │              51185 │              22204 │         43.38 │
│ 2019-04     │              49884 │              21607 │         43.31 │
│ 2019-05     │              48645 │              20657 │         42.46 │
│ 2019-06     │              52127 │  

In [125]:
con.sql("""
    SELECT
        COUNT(eligible_90d) AS eligible_customers,
        SUM(ret_90d) AS retained_customers,
        ROUND(100.0 * AVG(ret_90d),2) AS retention_90d
    FROM customer_lifecycle
    WHERE eligible_90d= 1
   
   
    
""").show()

┌────────────────────┬────────────────────┬───────────────┐
│ eligible_customers │ retained_customers │ retention_90d │
│       int64        │       int128       │    double     │
├────────────────────┼────────────────────┼───────────────┤
│            1291147 │             607798 │         47.07 │
└────────────────────┴────────────────────┴───────────────┘



In [126]:
con.execute("""
    CREATE OR REPLACE TABLE mart_cohort_retention_90d AS
    SELECT
        strftime(first_date, '%Y-%m') AS first_month,
        COUNT(eligible_90d) AS eligible_customers,
        SUM(ret_90d) AS retained_customers,
        ROUND(100.0 * AVG(ret_90d),2) AS retention_90d
    FROM customer_lifecycle
    WHERE eligible_90d = 1
    GROUP BY first_month
    ORDER BY first_month
    
""")

In [127]:
con.sql("""
    SELECT
        COUNT(*) AS cohort_months,
        MIN(first_month) AS first_cohort,
        MAX(first_month) AS last_cohort,
        SUM(eligible_customers) AS total_eligible,
        SUM(retained_customers) AS total_retained
    FROM mart_cohort_retention_90d
""").show()

┌───────────────┬──────────────┬─────────────┬────────────────┬────────────────┐
│ cohort_months │ first_cohort │ last_cohort │ total_eligible │ total_retained │
│     int64     │   varchar    │   varchar   │     int128     │     int128     │
├───────────────┼──────────────┼─────────────┼────────────────┼────────────────┤
│            22 │ 2018-09      │ 2020-06     │        1291147 │         607798 │
└───────────────┴──────────────┴─────────────┴────────────────┴────────────────┘



In [131]:
con.execute("""
    CREATE OR REPLACE VIEW v_cohort_retention_90d AS
    SELECT *,
      CASE
       WHEN first_month < '2019-01' THEN 'start_boundary'
       WHEN first_month > '2020-05' THEN 'end_partial'
       ELSE 'main_analysis'
      END AS cohort_window
    FROM mart_cohort_retention_90d
""") 

In [135]:
con.sql("""
    SELECT
        cohort_window,
        COUNT(cohort_window) AS cohort_month
    FROM v_cohort_retention_90d
    GROUP BY cohort_window
    ORDER BY cohort_window
""").show()

┌────────────────┬──────────────┐
│ cohort_window  │ cohort_month │
│    varchar     │    int64     │
├────────────────┼──────────────┤
│ end_partial    │            1 │
│ main_analysis  │           17 │
│ start_boundary │            4 │
└────────────────┴──────────────┘



In [139]:
con.sql("""
    SELECT
        COUNT(*) AS cohort_months,
        SUM(eligible_customers) AS eligible_customers,
        SUM(retained_customers) AS retained_customers,
        ROUND( SUM(retained_customers)/SUM(eligible_customers) * 100.0 ,2) AS weighted_retention,
        MIN(retention_90d) AS min_monthly_retention,
        MAX(retention_90d) AS max_monthly_retention
    FROM v_cohort_retention_90d
    WHERE cohort_window = 'main_analysis'
""").show()

┌───────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────────┬───────────────────────┐
│ cohort_months │ eligible_customers │ retained_customers │ weighted_retention │ min_monthly_retention │ max_monthly_retention │
│     int64     │       int128       │       int128       │       double       │        double         │        double         │
├───────────────┼────────────────────┼────────────────────┼────────────────────┼───────────────────────┼───────────────────────┤
│            17 │             685335 │             254739 │              37.17 │                 29.78 │                 43.38 │
└───────────────┴────────────────────┴────────────────────┴────────────────────┴───────────────────────┴───────────────────────┘



## 90-Day Cohort Retention Analysis

Customers were grouped by the month of their first observed purchase. The 90-day retention rate measures the percentage of eligible customers who made a second observed purchase within 90 days. Customers without 90 complete days of follow-up were excluded from the denominator.

The complete eligible population produced a 90-day retention rate of 47.07%. However, the first four cohorts were affected by the start of the observation window: customers appearing near the beginning of the dataset may have purchased before the available data period. The June 2020 cohort was also incomplete because only customers whose first observed purchase occurred by June 24 had sufficient follow-up.

For the main analysis, the comparable window was restricted to January 2019 through May 2020. Across these 17 monthly cohorts, 254,739 of 685,335 eligible customers made a second observed purchase within 90 days, resulting in a weighted retention rate of 37.17%.

Monthly retention ranged from 29.78% to 43.38%. The rate declined from 42.90% in January 2019 to 36.34% in May 2020, a decrease of 6.56 percentage points, although the pattern was not consistently downward in every month.

These results describe first-observed purchase cohorts rather than confirmed new-customer acquisition cohorts. The observed differences should not be interpreted as causal effects without further statistical analysis or experimental evidence.


In [149]:
con.sql("""
SELECT
    customer_id ,
    first_date,
    second_date,
    days_to_second,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE eligible_90d= 1
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
ORDER BY days_to_second
LIMIT 20
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬─────────────┬────────────────┬─────────┬─────────┐
│                           customer_id                            │ first_date │ second_date │ days_to_second │ ret_30d │ ret_60d │
│                             varchar                              │    date    │    date     │     int64      │  int32  │  int32  │
├──────────────────────────────────────────────────────────────────┼────────────┼─────────────┼────────────────┼─────────┼─────────┤
│ ed8aca689772e7cdb79877f9761a2ceb05d52673964630885c16f4f9fe21803e │ 2019-09-24 │ 2019-09-25  │              1 │       1 │       1 │
│ ee3acd7a9df0643a87d3e40cf5607d8caeb545f014f9a03d3df1f4bebd610d21 │ 2019-02-05 │ 2019-02-06  │              1 │       1 │       1 │
│ f18be272a8bd8b0122e06f53e343422407b0f57c6ca03c069dc2cf25f330f6b1 │ 2019-05-22 │ 2019-05-23  │              1 │       1 │       1 │
│ f284c75d0f4a4fc80a02eefea9615bdb731d1279920f258aa30d33072b2caef4 │ 

In [153]:
con.sql("""
SELECT DISTINCT
    days_to_second,
    ret_90d,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE days_to_second IN (29, 30, 31, 59, 60, 61, 89, 90, 91)
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
ORDER BY days_to_second
LIMIT 20
""").show()

┌────────────────┬─────────┬─────────┬─────────┐
│ days_to_second │ ret_90d │ ret_30d │ ret_60d │
│     int64      │  int32  │  int32  │  int32  │
├────────────────┼─────────┼─────────┼─────────┤
│             29 │       1 │       1 │       1 │
│             30 │       1 │       1 │       1 │
│             31 │       1 │       0 │       1 │
│             59 │       1 │       0 │       1 │
│             60 │       1 │       0 │       1 │
│             61 │       1 │       0 │       0 │
│             89 │       1 │       0 │       0 │
│             90 │       1 │       0 │       0 │
│             91 │       0 │       0 │       0 │
└────────────────┴─────────┴─────────┴─────────┘



In [164]:
con.sql("""
WITH classified_customers AS (
SELECT
    days_to_second,
    customer_id,
    first_date,
    ret_90d,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE eligible_90d= 1
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
)

SELECT 
    COUNT(*) AS eligible_customers,
    SUM(ret_30d) AS retained_30d,
    SUM(ret_60d) AS retained_60d,
    SUM(ret_90d) AS retained_90d,
    ROUND(
    100.0 * SUM(ret_30d) / COUNT(*),
    2) AS retention_30d,
    ROUND(
    100.0 * SUM(ret_60d) / COUNT(*),
    2 )AS retention_60d,
    ROUND(
    100.0 * SUM(ret_90d) / COUNT(*),
    2 )AS retention_90d
FROM classified_customers
ORDER BY retained_30d,
         retained_60d,
         retained_90d
""").show()


┌────────────────────┬──────────────┬──────────────┬──────────────┬───────────────┬───────────────┬───────────────┐
│ eligible_customers │ retained_30d │ retained_60d │ retained_90d │ retention_30d │ retention_60d │ retention_90d │
│       int64        │    int128    │    int128    │    int128    │    double     │    double     │    double     │
├────────────────────┼──────────────┼──────────────┼──────────────┼───────────────┼───────────────┼───────────────┤
│             685335 │       149800 │       210844 │       254739 │         21.86 │         30.77 │         37.17 │
└────────────────────┴──────────────┴──────────────┴──────────────┴───────────────┴───────────────┴───────────────┘



In [165]:
con.execute("""
WITH classified_customers AS (
SELECT
    days_to_second,
    customer_id,
    first_date,
    ret_90d,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE eligible_90d= 1
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
)

SELECT 
    COUNT(*) AS eligible_customers,
    SUM(ret_30d) AS retained_30d,
    SUM(ret_60d) AS retained_60d,
    SUM(ret_90d) AS retained_90d,
    ROUND(
    100.0 * SUM(ret_30d) / COUNT(*),
    2) AS retention_30d,
    ROUND(
    100.0 * SUM(ret_60d) / COUNT(*),
    2 )AS retention_60d,
    ROUND(
    100.0 * SUM(ret_90d) / COUNT(*),
    2 )AS retention_90d
FROM classified_customers
ORDER BY retained_30d,
         retained_60d,
         retained_90d
""")

In [169]:
con.sql("""
    SELECT 
        COUNT(*) AS retained_customers,
        MEDIAN(days_to_second) AS median_days_to_second
    FROM customer_lifecycle
    WHERE eligible_90d = 1
    AND ret_90d = 1
    AND first_date >= DATE '2019-01-01'
    AND first_date < DATE '2020-06-01'
""").show()

┌────────────────────┬───────────────────────┐
│ retained_customers │ median_days_to_second │
│       int64        │        double         │
├────────────────────┼───────────────────────┤
│             254739 │                  22.0 │
└────────────────────┴───────────────────────┘



# Among customers in the main comparable cohort window who made a second observed purchase within 90 days, the median time to the second purchase was 22 days. Of all eligible customers, 21.86% returned within 30 days, 30.77% within 60 days, and 37.17% within 90 days. This indicates that repeat purchasing was concentrated in the first month, although a substantial share of 90-day repeat customers returned between days 31 and 90.

In [171]:
con.sql("""
WITH main_population AS (
SELECT
    days_to_second,
    customer_id,
    first_date,
    ret_90d,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE eligible_90d= 1
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
)
SELECT
    COUNT(*) AS eligible_customers,
    SUM(ret_30d) AS retained_30d,
    ROUND(AVG(ret_30d)*100.0,2) AS ret_30d_pct,
    SUM(ret_60d) AS retained_60d,
    ROUND(AVG(ret_60d) * 100.0,2) AS ret_60d_pct,
    SUM(ret_90d) AS retained_90d,
    ROUND(AVG(ret_90d)* 100.0,2) AS ret_90d_pct,
    MEDIAN(
       CASE
         WHEN ret_90d =1 THEN days_to_second
         ELSE NULL
         END) AS median_days_to_second
FROM main_population 
""").show()

┌────────────────────┬──────────────┬─────────────┬──────────────┬─────────────┬──────────────┬─────────────┬───────────────────────┐
│ eligible_customers │ retained_30d │ ret_30d_pct │ retained_60d │ ret_60d_pct │ retained_90d │ ret_90d_pct │ median_days_to_second │
│       int64        │    int128    │   double    │    int128    │   double    │    int128    │   double    │        double         │
├────────────────────┼──────────────┼─────────────┼──────────────┼─────────────┼──────────────┼─────────────┼───────────────────────┤
│             685335 │       149800 │       21.86 │       210844 │       30.77 │       254739 │       37.17 │                  22.0 │
└────────────────────┴──────────────┴─────────────┴──────────────┴─────────────┴──────────────┴─────────────┴───────────────────────┘



### Retention velocity: 30, 60 and 90 days

### Retention velocity: 30, 60 and 90 days

The analysis includes 685,335 customers whose first observed purchase occurred between January 2019 and May 2020. The cumulative second-purchase rate increased from 21.86% within 30 days to 30.77% within 60 days and 37.17% within 90 days. Among customers who made a second observed purchase within 90 days, the median time to that purchase was 22 days. These results describe the timing and frequency of observed repeat purchases, but they do not establish what caused customers to return.


In [172]:
customers_relation = con.read_csv(str(data_dir / "customers.csv"))
customers_relation.create_view("customers", replace=True)

articles_relation = con.read_csv(str(data_dir / "articles.csv"))
articles_relation.create_view("articles", replace=True)

┌────────────┬──────────────┬───────────────────────────┬─────────────────┬───────────────────┬────────────────────┬─────────────────────────┬───────────────────────────┬───────────────────┬───────────────────┬───────────────────────────┬─────────────────────────────┬────────────────────────────┬──────────────────────────────┬───────────────┬───────────────────────┬────────────┬────────────────────────┬────────────────┬──────────────────┬────────────┬────────────────────────────────┬──────────────────┬────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ article_id │ product_code │         prod_name         │ product_type_no │ product_type_name │ product_group_name │ graphical_appearance_no │ graphical_appearance_name │ colour_group_code │ colour_group_name │ perceived_colour_

In [173]:
con.sql("""
    SELECT
        (SELECT COUNT(*) FROM customers) AS customer_rows,
        (SELECT COUNT(*) FROM articles) AS article_rows
""").show()

┌───────────────┬──────────────┐
│ customer_rows │ article_rows │
│     int64     │    int64     │
├───────────────┼──────────────┤
│       1371980 │       105542 │
└───────────────┴──────────────┘



In [176]:
con.sql("""
DESCRIBE customers
""").show()

┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ FN                     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Active                 │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ club_member_status     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ fashion_news_frequency │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ age                    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ postal_code            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [178]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(customer_id) AS populated_ids,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) - COUNT(customer_id) AS missing_ids,
    COUNT(customer_id) - COUNT(DISTINCT customer_id) AS duplicate_cust_rows
FROM customers
""").show()

┌────────────┬───────────────┬──────────────────┬─────────────┬─────────────────────┐
│ total_rows │ populated_ids │ unique_customers │ missing_ids │ duplicate_cust_rows │
│   int64    │     int64     │      int64       │    int64    │        int64        │
├────────────┼───────────────┼──────────────────┼─────────────┼─────────────────────┤
│    1371980 │       1371980 │          1371980 │           0 │                   0 │
└────────────┴───────────────┴──────────────────┴─────────────┴─────────────────────┘



In [180]:
con.sql("""
SELECT
    COUNT(*) - COUNT(age) AS missing_age,
    ROUND(
    100.0 * (COUNT(*) - COUNT(age)) / COUNT(*),
    2) AS missing_age_pct,
    COUNT(*) - COUNT( club_member_status) AS missing_status,
    ROUND(
    100.0 * (COUNT(*) - COUNT( club_member_status)) / COUNT(*),
    2) AS missing_status_pct,
    COUNT(*) - COUNT(fashion_news_frequency) AS missing_new_freq,
    ROUND(
    100.0 * (COUNT(*) - COUNT( fashion_news_frequency)) / COUNT(*),
    2) AS missing_new_freq_pct,
    COUNT(*) - COUNT(FN) AS missing_fn,
    COUNT(*) - COUNT(active) AS missing_active
FROM customers
""").show()
    

┌─────────────┬─────────────────┬────────────────┬────────────────────┬──────────────────┬──────────────────────┬────────────┬────────────────┐
│ missing_age │ missing_age_pct │ missing_status │ missing_status_pct │ missing_new_freq │ missing_new_freq_pct │ missing_fn │ missing_active │
│    int64    │     double      │     int64      │       double       │      int64       │        double        │   int64    │     int64      │
├─────────────┼─────────────────┼────────────────┼────────────────────┼──────────────────┼──────────────────────┼────────────┼────────────────┤
│       15861 │            1.16 │           6062 │               0.44 │            16009 │                 1.17 │     895050 │         907576 │
└─────────────┴─────────────────┴────────────────┴────────────────────┴──────────────────┴──────────────────────┴────────────┴────────────────┘



In [181]:
con.sql("""
SELECT
    FN,
    COUNT(*) AS customers
FROM customers
GROUP BY FN
ORDER BY FN DESC
""").show()

┌────────┬───────────┐
│   FN   │ customers │
│ double │   int64   │
├────────┼───────────┤
│    1.0 │    476930 │
│   NULL │    895050 │
└────────┴───────────┘



### Customer identifier and missing-value audit

The `customers` table contains 1,371,980 rows and the same number of unique, non-null customer IDs. Therefore, its observed grain is one row per customer, and `customer_id` can be used as the join key without creating duplicate customer records.

The `age`, `club_member_status`, and `fashion_news_frequency` columns have less than 1.2% missing values and can be considered for customer segmentation. In contrast, `FN` and `Active` have substantially higher missingness.

The `FN` column contains only `1.0` and `NULL`: 476,930 customers have the value `1.0`, while 895,050 have no recorded value. This suggests that `FN` is stored as a sparse binary indicator rather than a continuous numerical variable. However, `NULL` will be interpreted as “not indicated or not recorded,” rather than a confirmed negative value. Because the field may represent a customer snapshot rather than their status at the first observed purchase, any relationship with retention will be treated as descriptive and not causal.


# Inspect Active values to determine whether the column is encoded as a sparse binary indicator

In [183]:
con.sql("""
SELECT
    active,
    COUNT(*) AS customers
FROM customers
GROUP BY active
ORDER BY active DESC
""").show()

┌────────┬───────────┐
│ Active │ customers │
│ double │   int64   │
├────────┼───────────┤
│    1.0 │    464404 │
│   NULL │    907576 │
└────────┴───────────┘



The `Active` column follows the same sparse binary pattern: 464,404 customers are marked with `1.0`, while 907,576 have a null value. Because both `FN` and `Active` have limited coverage, ambiguous null values, and no historical timestamp, they will not be used as primary segmentation variables. The better-documented and more complete `fashion_news_frequency` field will be evaluated instead.

In [184]:
con.sql("""
SELECT
    fashion_news_frequency,
    COUNT(*) AS customers
FROM customers
GROUP BY fashion_news_frequency
ORDER BY fashion_news_frequency DESC
""").show()

┌────────────────────────┬───────────┐
│ fashion_news_frequency │ customers │
│        varchar         │   int64   │
├────────────────────────┼───────────┤
│ Regularly              │    477416 │
│ None                   │         2 │
│ NONE                   │    877711 │
│ Monthly                │       842 │
│ NULL                   │     16009 │
└────────────────────────┴───────────┘



# Standardize Fashion News categories into actionable CRM segments

In [189]:
con.sql("""
SELECT
    COUNT(*) AS customers,
    CASE 
      WHEN fashion_news_frequency IS  NULL THEN 'Unknown'
      WHEN UPPER(fashion_news_frequency) = 'NONE' THEN 'Not subscribed'
      WHEN fashion_news_frequency IN ('Regularly', 'Monthly')
      THEN 'Subscribed'
      ELSE 'Other'
      END AS news_segment
FROM customers
GROUP BY news_segment
ORDER BY customers DESC
""").show()

┌───────────┬────────────────┐
│ customers │  news_segment  │
│   int64   │    varchar     │
├───────────┼────────────────┤
│    877713 │ Not subscribed │
│    478258 │ Subscribed     │
│     16009 │ Unknown        │
└───────────┴────────────────┘



#### Fashion News frequency

The original `fashion_news_frequency` column contained inconsistent labels, including `NONE` and `None`, and a very small `Monthly` category. The values were standardized into three CRM-oriented segments: `Subscribed`, `Not subscribed`, and `Unknown`. The transformation preserved all 1,371,980 customer records. This variable will be used descriptively because the dataset does not indicate whether the recorded subscription status was already valid at the customer's first observed purchase.

In [191]:
con.sql("""
SELECT
    club_member_status,
    COUNT(*) AS customers
FROM customers
GROUP BY  club_member_status
ORDER BY  club_member_status DESC
""").show()

┌────────────────────┬───────────┐
│ club_member_status │ customers │
│      varchar       │   int64   │
├────────────────────┼───────────┤
│ PRE-CREATE         │     92960 │
│ LEFT CLUB          │       467 │
│ ACTIVE             │   1272491 │
│ NULL               │      6062 │
└────────────────────┴───────────┘



#### Club membership status

Most customers are classified as `ACTIVE` (92.75%), while `PRE-CREATE` represents 6.78%, `LEFT CLUB` only 0.03%, and 0.44% have an unknown status. For analysis, `PRE-CREATE` and `LEFT CLUB` will be combined into a broader `Not active member` segment, while null values will remain `Unknown`.

Because the table does not provide a timestamp for membership status, the recorded status may have been assigned after the first observed purchase. Therefore, any relationship between membership and retention will be interpreted as descriptive rather than causal.

# Inspect the age range and distribution before creating age segments

In [192]:
con.sql("""
SELECT
    MIN(age) AS min_age,
    QUANTILE_CONT(age, 0.25) AS p25_age,
    MEDIAN(age) AS median_age,
    QUANTILE_CONT(age, 0.75) AS p75_age,
    MAX(age) AS max_age
FROM customers
""").show()

┌─────────┬─────────┬────────────┬─────────┬─────────┐
│ min_age │ p25_age │ median_age │ p75_age │ max_age │
│  int64  │ double  │   double   │ double  │  int64  │
├─────────┼─────────┼────────────┼─────────┼─────────┤
│      16 │    24.0 │       32.0 │    49.0 │      99 │
└─────────┴─────────┴────────────┴─────────┴─────────┘



# Create interpretable age groups and validate their population sizes

In [195]:
con.sql("""
SELECT
    COUNT(*) AS customers,
    CASE
      WHEN age IS NULL THEN 'Unknown'
      WHEN age BETWEEN 16 AND 24 THEN '16-24'
      WHEN age BETWEEN 25 AND 34 THEN '25-34'
      WHEN age BETWEEN 35 AND 49 THEN '35-49'
      WHEN age >= 50 THEN '50+'
      ELSE 'Invalid'
    END AS age_group
FROM customers
GROUP BY age_group
ORDER BY customers DESC
""").show()

┌───────────┬───────────┐
│ customers │ age_group │
│   int64   │  varchar  │
├───────────┼───────────┤
│    393292 │ 25-34     │
│    357169 │ 16-24     │
│    317992 │ 50+       │
│    287666 │ 35-49     │
│     15861 │ Unknown   │
└───────────┴───────────┘



#### Customer age

Customer ages range from 16 to 99, with a median of 32. No implausible age values were identified. Age was grouped into four interpretable segments: `16-24`, `25-34`, `35-49`, and `50+`. Customers with missing age information were retained in an `Unknown` group. The transformation preserved all 1,371,980 customer records, and every valid age segment contains a sufficiently large population for comparison.

In [196]:
# Inspect the article schema and verify the join-key data type

con.sql("""
    DESCRIBE articles
""").show()

┌───────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │ column_type │  null   │   key   │ default │  extra  │
│          varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ article_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_code              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ prod_name                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_type_no           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ product_type_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_group_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ graphical_appearance_no   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ graphical_appearance_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

In [201]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(article_id) AS populated_ids,
    COUNT(DISTINCT article_id) AS unique_articles,
    COUNT(*) - COUNT(article_id) AS missing_ids ,
    COUNT(article_id) - COUNT(DISTINCT article_id) AS duplicate_art_rows,
    COUNT(*) - COUNT(product_group_name ) AS missing_product,
    ROUND(
    100.0 * (COUNT(*) - COUNT(product_group_name)) / COUNT(*),
    2) AS missing_product_groups_pct

FROM articles
""").show()

┌────────────┬───────────────┬─────────────────┬─────────────┬────────────────────┬─────────────────┬────────────────────────────┐
│ total_rows │ populated_ids │ unique_articles │ missing_ids │ duplicate_art_rows │ missing_product │ missing_product_groups_pct │
│   int64    │     int64     │      int64      │    int64    │       int64        │      int64      │           double           │
├────────────┼───────────────┼─────────────────┼─────────────┼────────────────────┼─────────────────┼────────────────────────────┤
│     105542 │        105542 │          105542 │           0 │                  0 │               0 │                        0.0 │
└────────────┴───────────────┴─────────────────┴─────────────┴────────────────────┴─────────────────┴────────────────────────────┘



#### Articles table audit

The `articles` table contains 105,542 rows and the same number of unique, non-null article IDs. Therefore, its observed grain is one row per article, and `article_id` can be used as a join key. The selected `product_group_name` field has complete coverage. Both `articles.article_id` and `transactions.article_id` are stored as `VARCHAR`, so no data-type conversion is required before joining them.

# Validate article join coverage and identify unmatched transaction records

In [202]:
con.sql("""
SELECT
    COUNT(*) AS transactions_rows,
    SUM(
    CASE
        WHEN a.article_id IS NULL THEN 1
        ELSE 0
    END) AS unmatched_transaction_rows,
    COUNT(
    DISTINCT CASE
        WHEN a.article_id IS NULL THEN t.article_id
    END) AS unmatched_article_ids
FROM transactions AS t
LEFT JOIN articles AS a
ON t.article_id = a.article_id
""").show()


┌───────────────────┬────────────────────────────┬───────────────────────┐
│ transactions_rows │ unmatched_transaction_rows │ unmatched_article_ids │
│       int64       │           int128           │         int64         │
├───────────────────┼────────────────────────────┼───────────────────────┤
│          31788324 │                          0 │                     0 │
└───────────────────┴────────────────────────────┴───────────────────────┘



#### Article join coverage

All 31,788,324 transaction rows matched a valid record in the article catalogue. No unmatched article IDs were identified, resulting in 100% join coverage. Therefore, `product_group_name` can be added to transaction records without losing purchases due to missing catalogue information.

# Identify article lines belonging to each customer's first observed purchase date

In [211]:
con.sql("""
SELECT
    l.customer_id,
    l.first_date,
    COUNT(*) AS n_lines,
    COUNT(DISTINCT t.article_id) AS n_articles,
    COUNT(DISTINCT t.sales_channel_id) AS n_channels,
    COUNT(DISTINCT a.product_group_name) AS n_groups
FROM customer_lifecycle AS l
JOIN transactions AS t
ON l.customer_id = t.customer_id
LEFT JOIN articles AS a
ON t.article_id = a.article_id
WHERE eligible_90d= 1
AND first_date >= '2019-01-01'
AND first_date < '2020-06-01'
GROUP BY l.customer_id,
         l.first_date
LIMIT 20
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬─────────┬────────────┬────────────┬──────────┐
│                           customer_id                            │ first_date │ n_lines │ n_articles │ n_channels │ n_groups │
│                             varchar                              │    date    │  int64  │   int64    │   int64    │  int64   │
├──────────────────────────────────────────────────────────────────┼────────────┼─────────┼────────────┼────────────┼──────────┤
│ 4dfb7f7f28f6dc5d83330a5ced17d55f1aea070525ed1c40b6ca0ac90459353b │ 2020-03-16 │      20 │         19 │          1 │        6 │
│ 4064fe4984cb2d7b969c041565d32e6ab0e5b4682709f52f9787424ba780053a │ 2019-03-24 │      25 │         24 │          2 │        7 │
│ 4c1b2a54d745c9a1cae5a0566e49b89455617200954d81e82a70d10c1dcbadf6 │ 2019-01-11 │      44 │         33 │          2 │        5 │
│ 9b85cb97ade7f195e3c0b40c8976955d105ea23b87595d7919315e25e04b0239 │ 2019-02-07 │      58 │      